In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from ydata_profiling import ProfileReport
import plotly.express as px
import itertools
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, precision_score, recall_score, f1_score
import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from catboost import CatBoostClassifier
from time import perf_counter
from sklearn.inspection import permutation_importance
from imblearn.over_sampling import SMOTENC
import shap


# Предобработка

In [ ]:
df_pipe1 = pd.read_excel('Протяженность, диаметр, давление.xlsx', sheet_name='внутрипромысловые трубопроводы')
df_pipe1.head()

In [ ]:
df_pipe1.info()

In [ ]:
example_addresses = (
    df_pipe1.groupby('id_регион')['Адрес (местоположение) ОС']
      .first()
      .reset_index()
)
example_addresses

In [ ]:
example_addresses.to_excel('data/Субъекты трубопроводов.xlsx', index=False)

In [ ]:
region_patterns = {
    r"адыгея": "Республика Адыгея",
    r"башкортостан": "Республика Башкортостан",
    r"дагестан": "Республика Дагестан",
    r"кабардино.*балкар": "Республика Кабардино-Балкария",
    r"калмык": "Республика Калмыкия",
    r"коми": "Республика Коми",
    r"мордов": "Республика Мордовия",               
    r"саха|якут": "Республика Саха (Якутия)",
    r"южн.*осет": "Республика Южная Осетия",
    r"татарстан": "Республика Татарстан",
    r"удмурт": "Республика Удмуртия",
    r"чечен": "Республика Чечня",
    r"чуваш": "Республика Чувашия",
    r"алтай.*край": "Алтайский край",
    r"краснодар": "Краснодарский край",
    r"ставроп": "Ставропольский край",
    r"примор": "Приморский край",
    r"камчат": "Камчатский край",
    "Москва": "Москва",
    r"амур": "Амурская область",                    
    r"перм": "Пермский край",
    r"хабаров": "Хабаровский край",
    r"архангел": "Архангельская область",
    r"астрахан": "Астраханская область",
    r"белгород": "Белгородская область",
    r"брянск": "Брянская область",
    r"владимир": "Владимирская область",
    r"волгоград": "Волгоградская область",
    r"вологод": "Вологодская область",
    r"воронеж": "Воронежская область",
    r"иванов": "Ивановская область",
    r"иркут": "Иркутская область",
    r"калининград": "Калининградская область",
    r"калуж": "Калужская область",
    r"кемеров": "Кемеровская область",
    r"киров": "Кировская область",
    r"костром": "Костромская область",
    r"курган": "Курганская область",
    r"курск": "Курская область",
    r"ленинград": "Ленинградская область",
    r"липец": "Липецкая область",
    r"москов": "Московская область",
    r"нижегород": "Нижегородская область",          
    r"новгород": "Новгородская область",
    r"новосиб": "Новосибирская область",
    r"омск": "Омская область",
    r"Оренбургская область": "Оренбургская область",
    r"Орлов": "Орловская область",
    r"пенз": "Пензенская область",
    r"псков": "Псковская область",
    r"ростов": "Ростовская область",
    r"рязан": "Рязанская область",
    r"самар": "Самарская область",
    r"сарат": "Саратовская область",
    r"сахалин": "Сахалинская область",
    r"свердлов": "Свердловская область",
    r"смолен": "Смоленская область",
    r"тамбов": "Тамбовская область",
    r"твер": "Тверская область",
    r"томск": "Томская область",
    r"туль": "Тульская область",
    r"тюмен": "Тюменская область",
    r"ульянов": "Ульяновская область",
    r"челябин": "Челябинская область",
    r"ярослав": "Ярославская область",
    r"ханты|югра|хмао": "Ханты-Мансийский автономный округ",
    r"янао|ямал": "Ямало-Ненецкий автономный округ",
    "Республика Южная Осетия": "Республика Южная Осетия",
}

overrides = {
    13: "Республика Мордовия",
    28: "Амурская область",
    52: "Нижегородская область",
    56: "Оренбургская область",
    57: "Орловская область"
}


def extract_region_text(address: str):
    """Пытается определить регион по тексту"""
    a = str(address).lower().strip()
    for pattern, name in region_patterns.items():
        if re.search(pattern, a):
            return name
    return "Неопределено"

df_pipe1['Субъект РФ'] = df_pipe1['Адрес (местоположение) ОС'].apply(extract_region_text)


def fill_region_by_id(row):
    """Если регион не найден, но есть id_регион в overrides — заполнить им"""
    if row['Субъект РФ'] == "Неопределено" and row['id_регион'] in overrides:
        return overrides[row['id_регион']]
    return row['Субъект РФ']

df_pipe1['Субъект РФ'] = df_pipe1.apply(fill_region_by_id, axis=1)

In [ ]:
df_pipe1 = df_pipe1.drop(columns=['Адрес (местоположение) ОС', 'id_регион'])
df_pipe1.head()

In [ ]:
df_pipe1['Субъект РФ'].value_counts()

In [ ]:
for col in df_pipe1.columns:
    try:
        uniques = df_pipe1[col].unique()
    except TypeError:
        uniques = df_pipe1[col].apply(lambda x: tuple(x) if isinstance(x, (list)) else x).unique()
    print(f'\n{col} ({len(uniques)} уникальных): {uniques}')

In [ ]:
df_pipe1.isna().sum()

In [ ]:
df_pipe1['Давление, Мпа'].describe()

In [ ]:
df_pipe1["Давление, Мпа"] = (
    df_pipe1.groupby(["Субъект РФ", "Способ укладки трубопровода"])["Давление, Мпа"]
      .transform(lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else x))
)
df_pipe1.isna().sum()

In [ ]:
df_pipe1['Давление, Мпа'].describe()

In [ ]:
df_pipe1["Давление, Мпа"] = df_pipe1["Давление, Мпа"].fillna(df_pipe1['Давление, Мпа'].describe()['50%'])
df_pipe1.isna().sum()

In [ ]:
df_pipe1['Давление, Мпа'].describe()

In [ ]:
df_pipe1[df_pipe1['Способ укладки трубопровода'].isna()]

In [ ]:
df_pipe1['Способ укладки трубопровода'].unique()

In [ ]:
df_pipe1['Способ укладки трубопровода'].describe()

In [ ]:
df_pipe1["Способ укладки трубопровода"] = (
    df_pipe1.groupby("Субъект РФ")["Способ укладки трубопровода"]
      .transform(lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else x))
)

In [ ]:
df_pipe1['Способ укладки трубопровода'].describe()

In [ ]:
df_pipe1.isna().sum()

In [ ]:
df_pipe1['Способ укладки трубопровода'] = df_pipe1['Способ укладки трубопровода'].fillna(df_pipe1['Способ укладки трубопровода'].describe()['top'])

In [ ]:
df_pipe1.isna().sum()

In [ ]:
df_pipe1[df_pipe1['Переоцененная стоимость (в ценах 01.01.2023), руб. без НДС'] == 0]

In [ ]:
df_pipe1["Переоцененная стоимость (в ценах 01.01.2023), руб. без НДС"] = df_pipe1.groupby(["Субъект РФ", "Способ укладки трубопровода"])["Переоцененная стоимость (в ценах 01.01.2023), руб. без НДС"].transform(
    lambda x: x.replace(0, pd.NA)  
).transform(
    lambda x: x.fillna(x.median())
)

In [ ]:
df_pipe1[df_pipe1['Переоцененная стоимость (в ценах 01.01.2023), руб. без НДС'] == 0]

In [ ]:
df_pipe1.duplicated().sum()

In [ ]:
today = pd.to_datetime("2023-01-01")

df_pipe1["Лет в эксплуатации"] = (
    today.year - df_pipe1["Дата ввода в экспл.в БУ"].dt.year
    - ((today.month < df_pipe1["Дата ввода в экспл.в БУ"].dt.month) |
       ((today.month == df_pipe1["Дата ввода в экспл.в БУ"].dt.month) &
        (today.day < df_pipe1["Дата ввода в экспл.в БУ"].dt.day)))
)

df_pipe1[["Дата ввода в экспл.в БУ", "Лет в эксплуатации"]].head()

In [ ]:
df_pipe1 = df_pipe1.drop(columns=['Дата ввода в экспл.в БУ', 'Инвентарный номер'])

In [ ]:
df_pipe1.columns = df_pipe1.columns.str.strip().str.lower()

In [ ]:
df_pipe1.columns

In [ ]:
df_pipe1.columns = ['класс', 'стоимость в ценах 2023 года без НДС, руб',
       'протяженность, м', 'диаметр трубопровода, мм', 'давление, мпа',
       'способ укладки трубопровода', 'субъект РФ', 'лет в эксплуатации']

In [ ]:
df_pipe1 = df_pipe1.apply(
    lambda col: col.astype(str).str.strip().replace(r'\s+', ' ', regex=True)
    if col.dtype == 'object' else col
)

In [ ]:
df_pipe2 = pd.read_excel('Протяженность, диаметр, давление.xlsx', sheet_name='газопроводы')
df_pipe2.head()

In [ ]:
df_pipe2.info()

In [ ]:
for col in df_pipe2.columns:
    try:
        uniques = df_pipe2[col].unique()
    except TypeError:
        uniques = df_pipe2[col].apply(lambda x: tuple(x) if isinstance(x, (list)) else x).unique()
    print(f'\n{col} ({len(uniques)} уникальных): {uniques}')

In [ ]:
df_pipe2.isna().sum()

In [ ]:
df_pipe2['Давление, Мпа'].describe()

In [ ]:
df_pipe2["Давление, Мпа"] = df_pipe2["Давление, Мпа"].fillna(df_pipe2['Давление, Мпа'].describe()['50%'])
df_pipe2.isna().sum()

In [ ]:
df_pipe2['Давление, Мпа'].describe()

In [ ]:
df_pipe2.duplicated().sum()

In [ ]:
today = pd.to_datetime("2023-01-01")

df_pipe2["Лет в эксплуатации"] = (
    today.year - df_pipe2["Дата ввода в экспл.в БУ"].dt.year
    - ((today.month < df_pipe2["Дата ввода в экспл.в БУ"].dt.month) |
       ((today.month == df_pipe2["Дата ввода в экспл.в БУ"].dt.month) &
        (today.day < df_pipe2["Дата ввода в экспл.в БУ"].dt.day)))
)

df_pipe2[["Дата ввода в экспл.в БУ", "Лет в эксплуатации"]].head()

In [ ]:
df_pipe2 = df_pipe2.drop(columns=['Дата ввода в экспл.в БУ', 'Инвентарный номер'])

In [ ]:
df_pipe2.columns = df_pipe2.columns.str.strip().str.lower()

In [ ]:
df_pipe2.columns

In [ ]:
df_pipe2.columns = ['класс', 'стоимость в ценах 2023 года без НДС, руб',
       'протяженность, м', 'диаметр трубопровода, мм', 'давление, мпа',
       'способ укладки трубопровода', 'субъект РФ', 'лет в эксплуатации']

In [ ]:
df_pipe2 = df_pipe2.apply(
    lambda col: col.astype(str).str.strip().replace(r'\s+', ' ', regex=True)
    if col.dtype == 'object' else col
)

# Объединение данных

In [ ]:
df_pipe = pd.concat([df_pipe1, df_pipe2], ignore_index=True)

In [ ]:
df_pipe.info()

In [ ]:
col = 'стоимость в ценах 2023 года без НДС, руб'

df_pipe[col] = (
    df_pipe[col]
    .astype(str)
    .str.replace(r'[^0-9.,]', '', regex=True)   
    .str.replace(',', '.', regex=False)         
    .astype(float)
)

print(df_pipe[col].head())
print(df_pipe[col].dtype)

In [ ]:
# Справочник «субъект РФ → код» хранится отдельно и не публикуется, чтобы коды на графиках нельзя было расшифровать.
import json
region_mapping = json.load(open("data/region_mapping.json", encoding="utf-8"))

In [ ]:
df_pipe['субъект РФ'] = df_pipe['субъект РФ'].map(region_mapping)

In [ ]:
df_pipe['субъект РФ'].unique()

In [ ]:
spusob_ukladki_mapping = {
    "Комбинированный": "Способ_01",
    "Наземный": "Способ_02",
    "Подземный": "Способ_03",
    "Надземный": "Способ_04"
}

In [ ]:
df_pipe["способ укладки трубопровода"] = df_pipe["способ укладки трубопровода"].map(spusob_ukladki_mapping)

In [ ]:
target_pipe = 'класс'
target_pipe_mapping = {'Внутрипромысловые трубопроводы': 0, 
                  'Газопроводы магистральные, отводы, лупинги': 1}

df_pipe[target_pipe] = df_pipe[target_pipe].map(target_pipe_mapping)

In [ ]:
df_pipe.to_excel('data/Общий файл (протяженность, диаметр, давление).xlsx', index=False)

In [ ]:
df_pipe = pd.read_excel('Общий файл (протяженность, диаметр, давление).xlsx')
df_pipe.head()

In [ ]:
df_pipe.info()

In [ ]:
X_pipe = df_pipe.drop(columns=[target_pipe])
y_pipe = df_pipe[target_pipe]

# Исследовательский анализ труб

In [ ]:
num_cols_pipe = ["стоимость в ценах 2023 года без НДС, руб", "протяженность, м", "диаметр трубопровода, мм", "давление, мпа", "лет в эксплуатации"]

plt.figure(figsize=(5, len(num_cols_pipe) * 3))

for i, col in enumerate(num_cols_pipe, 1):
    plt.subplot(len(num_cols_pipe), 1, i)
    sns.histplot(df_pipe[col], kde=True)
    plt.title(f'Распределение {col}')
    plt.xlabel(col)
    plt.ylabel('Количество')

plt.tight_layout()
plt.show()

In [ ]:
num_cols_pipe = [
    "стоимость в ценах 2023 года без НДС, руб",
    "протяженность, м",
    "диаметр трубопровода, мм",
    "давление, мпа",
    "лет в эксплуатации"
]

sns.pairplot(df_pipe[num_cols_pipe], diag_kind="kde")
plt.suptitle("Диаграмма рассеяния", y=1.02)
plt.show()

In [ ]:
num_cols_pipe = [
    "стоимость в ценах 2023 года без НДС, руб",
    "протяженность, м",
    "диаметр трубопровода, мм",
    "давление, мпа",
    "лет в эксплуатации"
]

df_corr = df_pipe[num_cols_pipe].copy()

rename_dict = {
    "стоимость в ценах 2023 года без НДС, руб": "Стоимость",
    "протяженность, м": "Протяженность",
    "диаметр трубопровода, мм": "Диаметр",
    "давление, мпа": "Давление",
    "лет в эксплуатации": "Возраст"
}

df_corr = df_corr.rename(columns=rename_dict)

corr_spearman = df_corr.corr(method="spearman")

plt.figure(figsize=(8, 6))
sns.heatmap(corr_spearman, annot=True, cmap="coolwarm", fmt=".3f")
plt.title("Корреляционная матрица")
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))

plt.subplot(2, 1, 1)
top10_ukladka = df_pipe['способ укладки трубопровода'].value_counts().nlargest(5).index
sns.countplot(x=df_pipe['способ укладки трубопровода'], order=top10_ukladka)
plt.title("Распределение способа укладки трубопровода")
plt.ylabel('Количество')

plt.subplot(2, 1, 2)
top10_subject_pipe = df_pipe['субъект РФ'].value_counts().nlargest(5).index
sns.countplot(x=df_pipe['субъект РФ'], order=top10_subject_pipe)
plt.title("Распределение по субъектам (топ 5)")
plt.ylabel('Количество')

plt.tight_layout()
plt.show()

In [ ]:
profile_pipe = ProfileReport(df_pipe, title="EDA отчёт", explorative=True)
report_pipe = ProfileReport(df_pipe, title="Первичный исследовательский анализ", explorative=True)
profile_pipe.to_file('data/eda_pipe_report.html')
report_pipe.to_notebook_iframe()

# Обучение

In [ ]:
num_pipe_features = X_pipe.select_dtypes(include=['int32', 'int64', 'float64']).columns.tolist()
cat_pipe_features = X_pipe.select_dtypes(include=['object', 'category']).columns.tolist()

numerical_pipe_transformer = StandardScaler()
categorical_pipe_transformer = OneHotEncoder(handle_unknown='ignore')
preprocessor_pipe = ColumnTransformer(
    transformers=[
        ('num', numerical_pipe_transformer, num_pipe_features),
        ('cat', categorical_pipe_transformer, cat_pipe_features)
    ])

In [ ]:
X_pipe_train, X_pipe_test, y_pipe_train, y_pipe_test = train_test_split(
    X_pipe, y_pipe, test_size=0.2, random_state=123, stratify=y_pipe)

In [ ]:
logreg_pipe = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',  
    random_state=123
)

pipeline_logreg_pipe = ImbPipeline([
    ('preprocessor', preprocessor_pipe),
    ('smote', SMOTE(random_state=123)),
    ('classifier', logreg_pipe)
])

param_grid_logreg_pipe = {
    'classifier__solver': ['liblinear', 'saga'], 
    'classifier__penalty': ['l1', 'l2', 'elasticnet'],  
    'classifier__C': [0.1, 1, 10],  
    'classifier__l1_ratio': [0, 0.5, 0.9], 
    'smote__k_neighbors': [2, 3, 5]  
}

cv_logreg_pipe = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

grid_logreg_pipe = GridSearchCV(
    pipeline_logreg_pipe,
    param_grid_logreg_pipe,
    scoring='roc_auc', 
    cv=cv_logreg_pipe,
    n_jobs=-1,
    verbose=1
)

grid_logreg_pipe.fit(X_pipe_train, y_pipe_train)

In [ ]:
print("Лучшие параметры:", grid_logreg_pipe.best_params_)
print("ROC-AUC (train):", grid_logreg_pipe.best_score_)

y_logreg_pipe_pred_proba = grid_logreg_pipe.predict_proba(X_pipe_test)[:, 1]
roc_logreg_pipe_test = roc_auc_score(y_pipe_test, y_logreg_pipe_pred_proba)
print("ROC-AUC (test):", roc_logreg_pipe_test)
y_logreg_pipe_pred = grid_logreg_pipe.predict(X_pipe_test)

cm = confusion_matrix(y_pipe_test, y_logreg_pipe_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_pipe_mapping.keys())
disp.plot(cmap='Blues', values_format='d')
plt.title("Матрица ошибок LogisticRegression")
plt.show()

print(classification_report(y_pipe_test, y_logreg_pipe_pred, labels = grid_logreg_pipe.classes_))

In [ ]:
knn_pipe = ImbPipeline([
    ('preprocessor', preprocessor_pipe),
    ('smote', SMOTE(random_state=123)),
    ('classifier', KNeighborsClassifier())
])

param_grid_knn_pipe = {
    'classifier__n_neighbors': [3, 5, 7, 10],
    'classifier__weights': ['uniform', 'distance'],
    'classifier__metric': ['euclidean', 'manhattan'],
    'smote__k_neighbors': [2, 3, 5]
}

cv_knn_pipe = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

grid_knn_pipe = GridSearchCV(
    knn_pipe,
    param_grid_knn_pipe,
    scoring='roc_auc',
    cv=cv_knn_pipe,
    n_jobs=-1,
    verbose=1
)

grid_knn_pipe.fit(X_pipe_train, y_pipe_train)

In [ ]:
print("Лучшие параметры:", grid_knn_pipe.best_params_)
print("ROC-AUC (train):", grid_knn_pipe.best_score_)

y_knn_pipe_pred_proba = grid_knn_pipe.predict_proba(X_pipe_test)[:, 1]
roc_knn_pipe_test = roc_auc_score(y_pipe_test, y_knn_pipe_pred_proba)
print("ROC-AUC (test):", roc_knn_pipe_test)

y_knn_pipe_pred = grid_knn_pipe.predict(X_pipe_test)

custom_labels = ["Трубопроводы", "Газопроводы"]
cm = confusion_matrix(y_pipe_test, y_knn_pipe_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=custom_labels)
disp.plot(cmap="Blues", values_format="d")
plt.title("Матрица ошибок KNN")
plt.show()

print(classification_report(y_pipe_test, y_knn_pipe_pred))

In [ ]:
dt_pipe = ImbPipeline(steps=[
    ('preprocessor', preprocessor_pipe),
    ('smote', SMOTE(random_state=123)),
    ('classifier', DecisionTreeClassifier(random_state=123))
])

param_grid_dt_pipe = {
    "classifier__max_depth": [5, 10, 15],
    "classifier__min_samples_split": [5, 10, 15],
    "classifier__min_samples_leaf": [2, 4, 6],
    "classifier__criterion": ["gini", "entropy"],
    "smote__k_neighbors": [2, 3, 5]
}

cv_dt_pipe = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

grid_dt_pipe = GridSearchCV(
    dt_pipe,
    param_grid_dt_pipe,
    scoring='roc_auc',
    cv=cv_dt_pipe,
    n_jobs=-1,
    verbose=1
)

grid_dt_pipe.fit(X_pipe_train, y_pipe_train)

In [ ]:
print("Лучшие параметры:", grid_dt_pipe.best_params_)
print("ROC-AUC (train):", grid_dt_pipe.best_score_)

y_dt_pipe_pred_proba = grid_dt_pipe.predict_proba(X_pipe_test)[:, 1]
roc_dt_pipe_test = roc_auc_score(y_pipe_test, y_dt_pipe_pred_proba)

print("ROC-AUC (test):", roc_dt_pipe_test)

y_dt_pipe_pred = grid_dt_pipe.predict(X_pipe_test)

cm = confusion_matrix(y_pipe_test, y_dt_pipe_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues')
plt.title("Матрица ошибок Decision Tree")
plt.show()

print(classification_report(y_pipe_test, y_dt_pipe_pred))

In [ ]:
num_features = num_pipe_features

cat_encoder = preproc.named_transformers_['cat']
cat_features = cat_encoder.get_feature_names_out(cat_pipe_features)

feature_names = np.concatenate([num_features, cat_features])

X_transformed = preproc.transform(X_pipe_test)

if hasattr(X_transformed, "toarray"):
    X_transformed = X_transformed.toarray()

X_transformed = pd.DataFrame(
    X_transformed,
    columns=feature_names)

explainer = shap.TreeExplainer(clf)

shap_values = explainer.shap_values(X_transformed)

shap_vals = shap_values[:, :, 1]

In [ ]:
scaler = StandardScaler()

X_train_scaled = X_pipe_train.copy()
X_test_scaled = X_pipe_test.copy()

X_train_scaled[num_pipe_features] = scaler.fit_transform(
    X_train_scaled[num_pipe_features]
)

X_test_scaled[num_pipe_features] = scaler.transform(
    X_test_scaled[num_pipe_features]
)

for c in cat_pipe_features:
    X_train_scaled[c] = X_train_scaled[c].astype(str)
    X_test_scaled[c] = X_test_scaled[c].astype(str)
    
cat_idx = [X_train_scaled.columns.get_loc(c) for c in cat_pipe_features]

smote_nc = SMOTENC(
    categorical_features=cat_idx,
    random_state=123
)

X_res, y_res = smote_nc.fit_resample(
    X_train_scaled,
    y_pipe_train
)

model_cb_pipe = CatBoostClassifier(
    random_state=123,
    loss_function="Logloss",
    eval_metric="AUC",
    verbose=0
)

param_grid_cb_pipe = {
    "depth": [3, 5, 10],
    "learning_rate": [0.03, 0.05],
    "iterations": [100, 200, 300],
    "l2_leaf_reg": [2, 3, 5]
}

cv_cb_pipe = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

grid_cb_pipe = GridSearchCV(
    estimator=model_cb_pipe,
    param_grid=param_grid_cb_pipe,
    scoring="roc_auc",
    cv=cv_cb_pipe,
    n_jobs=-1,
    verbose=1
)

grid_cb_pipe.fit(
    X_res,
    y_res,
    cat_features=cat_idx
)

In [ ]:
print("Лучшие параметры:", grid_cb_pipe.best_params_)
print("ROC-AUC (train):", grid_cb_pipe.best_score_)

y_cb_pipe_pred_proba = grid_cb_pipe.predict_proba(X_test_scaled)[:, 1]
roc_cb_pipe_test = roc_auc_score(y_pipe_test, y_cb_pipe_pred_proba)
print("ROC-AUC (test):", roc_cb_pipe_test)

y_cb_pipe_pred = grid_cb_pipe.predict(X_test_scaled)

cm = confusion_matrix(y_pipe_test, y_cb_pipe_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues')
plt.title("Матрица ошибок CatBoost")
plt.show()

print(classification_report(y_pipe_test, y_cb_pipe_pred))

In [ ]:
models_pipe_probs = {
    'KNN': y_knn_pipe_pred_proba,
    'Logreg': y_logreg_pipe_pred_proba,
    'Decision Tree': y_dt_pipe_pred_proba,
    'CatBoost': y_cb_pipe_pred_proba
}

fig = go.Figure()

for model_name, y_proba in models_pipe_probs.items():
    fpr, tpr, _ = roc_curve(y_pipe_test, y_proba)
    auc = roc_auc_score(y_pipe_test, y_proba)
    fig.add_trace(go.Scatter(
        x=fpr,
        y=tpr,
        mode='lines',
        name=f'{model_name} (AUC = {auc:.3f})',
        line=dict(width=3)
    ))
    
fig.add_trace(go.Scatter(
    x=[0,1],
    y=[0,1],
    mode='lines',
    line=dict(color='gray', dash='dash'),
    name='Random'
))

fig.update_layout(
    title='Сравнение ROC-кривых для всех моделей',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    template='plotly_white',
    height=750,
    width=750,
    font=dict(size=14),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.2,
        xanchor="center",
        x=0.5
    )
)

fig.show()

In [ ]:
results = []

def evaluate_model(
    name,
    model,
    X_train,
    y_train,
    X_test,
    y_test,
    cat_features=None
):
    start = perf_counter()

    if cat_features is not None:
        model.fit(X_train, y_train, cat_features=cat_features)
    else:
        model.fit(X_train, y_train)

    train_time = perf_counter() - start

    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)
    else:
        y_proba = None

    auc = None
    if y_proba is not None:
        if y_proba.ndim == 2 and y_proba.shape[1] == 2:
            auc = roc_auc_score(y_test, y_proba[:, 1])

        elif y_proba.ndim == 2 and y_proba.shape[1] > 2:
            auc = roc_auc_score(
                y_test,
                y_proba,
                multi_class="ovr",
                average="weighted"
            )
        else:
            auc = roc_auc_score(y_test, y_proba)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    results.append({
        "model": name,
        "train_time_sec": round(train_time, 3),
        "AUC": auc,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1
    })

In [ ]:
evaluate_model("Logistic Regression", grid_logreg_pipe, X_pipe_train, y_pipe_train, X_pipe_test, y_pipe_test)
evaluate_model("KNN", grid_knn_pipe, X_pipe_train, y_pipe_train, X_pipe_test, y_pipe_test)
evaluate_model("Decision Tree", grid_dt_pipe, X_pipe_train, y_pipe_train, X_pipe_test, y_pipe_test)
evaluate_model(
    name="CatBoost",
    model=grid_cb_pipe,
    X_train=X_res,
    y_train=y_res,
    X_test=X_test_scaled,
    y_test=y_pipe_test,
    cat_features=cat_idx
)

In [ ]:
df_metrics = pd.DataFrame(results)
df_metrics

In [ ]:
feature_names = (
    grid_logreg_pipe.best_estimator_["preprocessor"]
    .get_feature_names_out()
)

coefs_logreg = np.abs(grid_logreg_pipe.best_estimator_["classifier"].coef_).mean(axis=0)

fi_logreg = pd.DataFrame({
    "feature": feature_names,
    "importance": coefs_logreg
}).sort_values("importance", ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(fi_logreg.head(10)["feature"], fi_logreg.head(10)["importance"])
plt.title("Важность признаков - Logistic Regression")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
best_model = grid_knn_pipe.best_estimator_

preproc = best_model.named_steps["preprocessor"]
preproc.fit(X_pipe_train)

X_test_transformed = preproc.transform(X_pipe_test)

scoring_metric = "roc_auc" if len(np.unique(y_pipe_test)) == 2 else "roc_auc_ovr"

X_test_dense = X_test_transformed.toarray()

r = permutation_importance(
    estimator=best_model.named_steps["classifier"],  
    X=X_test_dense,
    y=y_pipe_test,
    n_repeats=10,
    random_state=123,
    scoring=scoring_metric
)

feature_names = preproc.get_feature_names_out()

fi = pd.DataFrame({
    "feature": feature_names,
    "importance": r.importances_mean
}).sort_values("importance", ascending=False)

plt.figure(figsize=(8, 8))
plt.barh(fi["feature"].head(10), fi["importance"].head(10))
plt.title("Пермутационная важность признаков — KNN")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
best_tree = grid_dt_pipe.best_estimator_
clf = best_tree.named_steps["classifier"]
preproc = best_tree.named_steps['preprocessor']

feature_names = preproc.get_feature_names_out()
importances = clf.feature_importances_

fi = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

plt.figure(figsize=(8, 6))
plt.barh(fi["feature"].head(10), fi["importance"].head(10))
plt.title("Важность признаков — Decision Tree")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
shap.summary_plot(
    shap_vals,
    X_transformed,
    plot_type="bar",
    show=True
)

In [ ]:
shap.summary_plot(
    shap_vals,
    X_transformed
)

In [ ]:
clf = grid_cb_pipe.best_estimator_
importances = clf.get_feature_importance()
feature_names = X_train_scaled.columns
fi = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)
plt.figure(figsize=(8, 6))
plt.barh(fi["feature"].head(10), fi["importance"].head(10))
plt.title("Важность признаков — CatBoost")
plt.gca().invert_yaxis()
plt.show()